# 07 — OA to Geography Aggregation, K=7

This notebook aggregates the preferred OA-level cluster model upward into LSOA21, MSOA21, LAD23, Ward25, LAD25, and LEP if available.

It uses two lookup files:

1. `oa21_to_wd25_lad25_eng_wal(may25).csv` — OA21 → Ward25 → LAD25.
2. `oa21_to_lsoa21_msoa21_lep_lad23_eng(april23).csv` — OA21 → LSOA21 → MSOA21 → LEP → LAD23.

Core rules:

- OA is the atomic unit.
- Cluster shares are aggregated by summing OA population by cluster.
- Census statistics are aggregated by summing counts first, then recalculating percentages.
- Ward outputs preserve `LAD25CD` and `LAD25NM` by grouping on LAD + ward columns together.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

## 1. Paths and configuration

This assumes the notebook sits in `Electoral_Tribes/notebooks` and your project root is `Electoral_Tribes`.

In [2]:
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
DATA_GEOGRAPHY = PROJECT_DIR / "data" / "geography"
OUTPUT_DIR = DATA_PROCESSED / "aggregations_v1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K = 7

ASSIGNMENT_PATH = (
    DATA_PROCESSED
    / "k_comparison_outputs_v1"
    / "oa_assignments"
    / f"k{K}_oa_cluster_assignments_v1.csv"
)

FEATURES_PATH = DATA_PROCESSED / "oa_derived_features_full_v1.csv"

WARD_LOOKUP_PATH = DATA_GEOGRAPHY / "oa21_to_wd25_lad25_eng_wal(may25).csv"
EXACT_LOOKUP_PATH = DATA_GEOGRAPHY / "oa21_to_lsoa21_msoa21_lep_lad23_eng(april23).csv"

print("Project folder:", PROJECT_DIR)
print("Processed data folder:", DATA_PROCESSED)
print("Geography folder:", DATA_GEOGRAPHY)
print("Output folder:", OUTPUT_DIR)

for path in [ASSIGNMENT_PATH, FEATURES_PATH, WARD_LOOKUP_PATH, EXACT_LOOKUP_PATH]:
    print(path.name, "exists:", path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

Project folder: c:\Users\keena\Documents\Electoral_Tribes
Processed data folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed
Geography folder: c:\Users\keena\Documents\Electoral_Tribes\data\geography
Output folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\aggregations_v1
k7_oa_cluster_assignments_v1.csv exists: True
oa_derived_features_full_v1.csv exists: True
oa21_to_wd25_lad25_eng_wal(may25).csv exists: True
oa21_to_lsoa21_msoa21_lep_lad23_eng(april23).csv exists: True


## 2. Helper functions

These functions standardise inconsistent lookup column names and provide reusable aggregation logic.

In [3]:
def norm_col(col: str) -> str:
    # Normalise a column name for robust matching.
    col = str(col).strip().upper()
    return re.sub(r"[^A-Z0-9]", "", col)


def find_col(df: pd.DataFrame, candidates=None, patterns=None, required=True, label="column"):
    # Find a column using normalised exact candidates first, then regex patterns.
    candidates = candidates or []
    patterns = patterns or []

    norm_map = {norm_col(c): c for c in df.columns}

    for cand in candidates:
        key = norm_col(cand)
        if key in norm_map:
            return norm_map[key]

    for pattern in patterns:
        rx = re.compile(pattern)
        for normed, original in norm_map.items():
            if rx.search(normed):
                return original

    if required:
        raise ValueError(
            f"Could not find {label}. Candidates={candidates}; patterns={patterns}; "
            f"available={list(df.columns)}"
        )

    return None


def safe_divide(numerator, denominator):
    # Vectorised division that returns NaN when denominator is zero.
    return np.where(denominator > 0, numerator / denominator, np.nan)


def add_area_metadata(df, geography_level, method, source_lookup):
    df = df.copy()
    df["geography_level"] = geography_level
    df["aggregation_method"] = method
    df["source_lookup"] = source_lookup
    return df


def build_group_cols(df, target_code, target_name=None, extra_group_cols=None):
    # Build grouping columns while preserving parent geography fields where needed.
    extra_group_cols = extra_group_cols or []
    group_cols = []

    for col in extra_group_cols:
        if col in df.columns and col not in group_cols:
            group_cols.append(col)

    if target_code not in group_cols:
        group_cols.append(target_code)

    if target_name is not None and target_name in df.columns and target_name not in group_cols:
        group_cols.append(target_name)

    return group_cols

## 3. Load OA cluster assignments and OA derived features

The assignment file is the output from the K comparison notebook. The feature file is used as the authoritative source for `total_residents` where available.

In [4]:
assignments_raw = pd.read_csv(ASSIGNMENT_PATH, low_memory=False)
features = pd.read_csv(FEATURES_PATH, low_memory=False)

print("Assignments shape:", assignments_raw.shape)
print("Assignments columns:", assignments_raw.columns.tolist())
print("Features shape:", features.shape)
print("Feature columns sample:", features.columns[:20].tolist())

oa_col = find_col(
    assignments_raw,
    ["OA21CD", "oa_code", "OA code", "Output Area Code"],
    patterns=[r"^OA.*CD$"],
    label="OA code in assignments"
)

cluster_col = find_col(
    assignments_raw,
    ["cluster_id", "cluster", "Cluster ID"],
    patterns=[r"CLUSTER"],
    label="cluster id"
)

pop_col = find_col(
    assignments_raw,
    ["population", "total_residents", "pop"],
    patterns=[r"POPULATION|TOTALRESIDENTS"],
    required=False,
    label="population"
)

assignments = assignments_raw.rename(columns={
    oa_col: "OA21CD",
    cluster_col: "cluster_id",
}).copy()

if pop_col is not None:
    assignments = assignments.rename(columns={pop_col: "population"})

assignments["OA21CD"] = assignments["OA21CD"].astype(str).str.strip()
assignments["cluster_id"] = assignments["cluster_id"].astype(int)

features_oa_col = find_col(
    features,
    ["OA21CD", "oa_code", "OA code", "geography code"],
    patterns=[r"^OA.*CD$"],
    label="OA code in features"
)

features = features.rename(columns={features_oa_col: "OA21CD"}).copy()
features["OA21CD"] = features["OA21CD"].astype(str).str.strip()

# Prefer total_residents from features as authoritative population.
if "total_residents" in features.columns:
    assignments = assignments.drop(columns=["population"], errors="ignore").merge(
        features[["OA21CD", "total_residents"]],
        on="OA21CD",
        how="left",
        validate="one_to_one"
    ).rename(columns={"total_residents": "population"})

if "population" not in assignments.columns:
    raise ValueError("No population column found. Expected either assignments.population or features.total_residents.")

assignments["population"] = pd.to_numeric(assignments["population"], errors="coerce")

print(assignments.head())
print("Assignment rows:", len(assignments))
print("Unique OAs:", assignments["OA21CD"].nunique())
print("Missing population:", assignments["population"].isna().sum())

Assignments shape: (188880, 3)
Assignments columns: ['oa_code', 'cluster_id', 'population']
Features shape: (188880, 153)
Feature columns sample: ['OA21CD', 'total_residents', 'household_residents', 'communal_establishment_residents', 'communal_establishment_pct', 'age_0_14_count', 'age_15_24_count', 'age_25_34_count', 'age_35_49_count', 'age_50_64_count', 'age_65_plus_count', 'age_0_14_pct', 'age_15_24_pct', 'age_25_34_pct', 'age_35_49_pct', 'age_50_64_pct', 'age_65_plus_pct', 'partnership_total', 'never_married_count', 'married_or_civil_partnership_count']
      OA21CD  cluster_id  population
0  E00000001           3         176
1  E00000003           3         256
2  E00000005           3         112
3  E00000007           3         144
4  E00000010           3         178
Assignment rows: 188880
Unique OAs: 188880
Missing population: 0


## 4. Load and standardise geography lookups

The ward lookup provides LAD25 fields. The exact-fit lookup provides LSOA21/MSOA21/LAD23 and optional LEP fields.

In [5]:
ward_lookup_raw = pd.read_csv(WARD_LOOKUP_PATH, low_memory=False)
exact_lookup_raw = pd.read_csv(EXACT_LOOKUP_PATH, low_memory=False)

print("Ward lookup shape:", ward_lookup_raw.shape)
print("Ward lookup columns:", ward_lookup_raw.columns.tolist())
print("Exact lookup shape:", exact_lookup_raw.shape)
print("Exact lookup columns:", exact_lookup_raw.columns.tolist())

Ward lookup shape: (188880, 8)
Ward lookup columns: ['OA21CD', 'WD25CD', 'WD25NM', 'WD25NMW', 'LAD25CD', 'LAD25NM', 'LAD25NMW', 'ObjectId']
Exact lookup shape: (178605, 12)
Exact lookup columns: ['OA21CD', 'LAD23CD', 'LAD23NM', 'LSOA21CD', 'LSOA21NM', 'MSOA21CD', 'MSOA21NM', 'LEP23CD1', 'LEP23NM1', 'LEP23CD2', 'LEP23NM2', 'ObjectId']


In [6]:
# OA21 -> WD25 -> LAD25
ward_lookup = ward_lookup_raw.copy()

ward_cols = {
    "OA21CD": find_col(
        ward_lookup,
        ["OA21CD", "OA21_CODE", "Output Area Code", "OA code"],
        patterns=[r"^OA21CD$", r"OUTPUTAREA.*2021.*CODE", r"^OA.*CODE$"],
        label="OA21CD in ward lookup"
    ),
    "WD25CD": find_col(
        ward_lookup,
        ["WD25CD", "WARD25CD", "Electoral Ward 2025 Code", "Ward Code"],
        patterns=[r"^WD25CD$", r"WARD.*2025.*CODE", r"ELECTORALWARD.*CODE"],
        label="WD25CD in ward lookup"
    ),
    "WD25NM": find_col(
        ward_lookup,
        ["WD25NM", "WARD25NM", "Electoral Ward 2025 Name", "Ward Name"],
        patterns=[r"^WD25NM$", r"WARD.*2025.*NAME", r"ELECTORALWARD.*NAME"],
        label="WD25NM in ward lookup"
    ),
    "LAD25CD": find_col(
        ward_lookup,
        ["LAD25CD", "Local Authority District 2025 Code", "LAD Code"],
        patterns=[r"^LAD25CD$", r"LOCALAUTHORITY.*2025.*CODE", r"LAD.*2025.*CODE"],
        label="LAD25CD in ward lookup"
    ),
    "LAD25NM": find_col(
        ward_lookup,
        ["LAD25NM", "Local Authority District 2025 Name", "LAD Name"],
        patterns=[r"^LAD25NM$", r"LOCALAUTHORITY.*2025.*NAME", r"LAD.*2025.*NAME"],
        label="LAD25NM in ward lookup"
    ),
}

ward_lookup = (
    ward_lookup[list(ward_cols.values())]
    .rename(columns={v: k for k, v in ward_cols.items()})
    .drop_duplicates(subset=["OA21CD"])
)

for col in ["OA21CD", "WD25CD", "WD25NM", "LAD25CD", "LAD25NM"]:
    ward_lookup[col] = ward_lookup[col].astype(str).str.strip()

print(ward_lookup.head())
print("Ward lookup unique OAs:", ward_lookup["OA21CD"].nunique())
print("Ward lookup rows:", len(ward_lookup))

      OA21CD     WD25CD    WD25NM    LAD25CD     LAD25NM
0  E00060547  E05013049  Victoria  E06000001  Hartlepool
1  E00060399  E05013049  Victoria  E06000001  Hartlepool
2  E00060390  E05013049  Victoria  E06000001  Hartlepool
3  E00060385  E05013049  Victoria  E06000001  Hartlepool
4  E00060397  E05013049  Victoria  E06000001  Hartlepool
Ward lookup unique OAs: 188880
Ward lookup rows: 188880


In [7]:
# OA21 -> LSOA21 -> MSOA21 -> LEP -> LAD23
exact_lookup = exact_lookup_raw.copy()

exact_cols = {
    "OA21CD": find_col(
        exact_lookup,
        ["OA21CD", "OA21_CODE", "Output Area Code", "OA code"],
        patterns=[r"^OA21CD$", r"OUTPUTAREA.*2021.*CODE", r"^OA.*CODE$"],
        label="OA21CD in exact lookup"
    ),
    "LSOA21CD": find_col(
        exact_lookup,
        ["LSOA21CD", "LSOA21_CODE", "LSOA code"],
        patterns=[r"^LSOA21CD$", r"LOWER.*SUPER.*OUTPUT.*2021.*CODE", r"LSOA.*2021.*CODE"],
        label="LSOA21CD in exact lookup"
    ),
    "LSOA21NM": find_col(
        exact_lookup,
        ["LSOA21NM", "LSOA21_NAME", "LSOA name"],
        patterns=[r"^LSOA21NM$", r"LOWER.*SUPER.*OUTPUT.*2021.*NAME", r"LSOA.*2021.*NAME"],
        label="LSOA21NM in exact lookup"
    ),
    "MSOA21CD": find_col(
        exact_lookup,
        ["MSOA21CD", "MSOA21_CODE", "MSOA code"],
        patterns=[r"^MSOA21CD$", r"MIDDLE.*SUPER.*OUTPUT.*2021.*CODE", r"MSOA.*2021.*CODE"],
        label="MSOA21CD in exact lookup"
    ),
    "MSOA21NM": find_col(
        exact_lookup,
        ["MSOA21NM", "MSOA21_NAME", "MSOA name"],
        patterns=[r"^MSOA21NM$", r"MIDDLE.*SUPER.*OUTPUT.*2021.*NAME", r"MSOA.*2021.*NAME"],
        label="MSOA21NM in exact lookup"
    ),
    "LAD23CD": find_col(
        exact_lookup,
        ["LAD23CD", "LAD23_CODE", "Local Authority District 2023 Code", "LAD code"],
        patterns=[r"^LAD23CD$", r"LOCALAUTHORITY.*2023.*CODE", r"LAD.*2023.*CODE"],
        label="LAD23CD in exact lookup"
    ),
    "LAD23NM": find_col(
        exact_lookup,
        ["LAD23NM", "LAD23_NAME", "Local Authority District 2023 Name", "LAD name"],
        patterns=[r"^LAD23NM$", r"LOCALAUTHORITY.*2023.*NAME", r"LAD.*2023.*NAME"],
        label="LAD23NM in exact lookup"
    ),
}

lep_cd_col = find_col(
    exact_lookup,
    ["LEP23CD", "LEPCD", "LEP_CODE"],
    patterns=[r"^LEP[0-9]*CD$", r"LOCALENTERPRISE.*CODE"],
    required=False,
    label="LEP code"
)
lep_nm_col = find_col(
    exact_lookup,
    ["LEP23NM", "LEPNM", "LEP_NAME"],
    patterns=[r"^LEP[0-9]*NM$", r"LOCALENTERPRISE.*NAME"],
    required=False,
    label="LEP name"
)

if lep_cd_col is not None:
    exact_cols["LEPCD"] = lep_cd_col
if lep_nm_col is not None:
    exact_cols["LEPNM"] = lep_nm_col

exact_lookup = (
    exact_lookup[list(exact_cols.values())]
    .rename(columns={v: k for k, v in exact_cols.items()})
    .drop_duplicates(subset=["OA21CD"])
)

for col in exact_lookup.columns:
    if col.endswith("CD") or col.endswith("NM"):
        exact_lookup[col] = exact_lookup[col].astype(str).str.strip()

print(exact_lookup.head())
print("Exact lookup unique OAs:", exact_lookup["OA21CD"].nunique())
print("Exact lookup rows:", len(exact_lookup))
print("LEP columns included:", [c for c in ["LEPCD", "LEPNM"] if c in exact_lookup.columns])

      OA21CD   LSOA21CD     LSOA21NM   MSOA21CD    MSOA21NM    LAD23CD LAD23NM
0  E00049234  E01009722  Dudley 027B  E02002026  Dudley 027  E08000027  Dudley
1  E00049235  E01009725  Dudley 031B  E02002030  Dudley 031  E08000027  Dudley
2  E00049236  E01009719  Dudley 027A  E02002026  Dudley 027  E08000027  Dudley
3  E00049237  E01009719  Dudley 027A  E02002026  Dudley 027  E08000027  Dudley
4  E00049238  E01009725  Dudley 031B  E02002030  Dudley 031  E08000027  Dudley
Exact lookup unique OAs: 178605
Exact lookup rows: 178605
LEP columns included: []


## 5. Build the OA geography-cluster base

This joins every clustered OA to both geography lookup routes. This table is the base for all later aggregation.

In [8]:
oa_base = (
    assignments[["OA21CD", "cluster_id", "population"]]
    .merge(exact_lookup, on="OA21CD", how="left", validate="one_to_one")
    .merge(ward_lookup, on="OA21CD", how="left", validate="one_to_one")
)

print("OA base shape:", oa_base.shape)
display(oa_base.head())

coverage = pd.DataFrame({
    "column": oa_base.columns,
    "missing_count": [oa_base[c].isna().sum() for c in oa_base.columns],
    "missing_pct": [oa_base[c].isna().mean() for c in oa_base.columns],
}).sort_values("missing_pct", ascending=False)

display(coverage)

coverage.to_csv(OUTPUT_DIR / f"k{K}_oa_geo_coverage_report_v1.csv", index=False)
oa_base.to_csv(OUTPUT_DIR / f"k{K}_oa_geo_cluster_base_v1.csv", index=False)

OA base shape: (188880, 13)


,OA21CD,cluster_id,population,LSOA21CD,LSOA21NM,MSOA21CD,MSOA21NM,LAD23CD,LAD23NM,WD25CD,WD25NM,LAD25CD,LAD25NM
0,E00000001,3,176,E01000001,City of London 001A,E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E09000001,City of London
1,E00000003,3,256,E01000001,City of London 001A,E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E09000001,City of London
2,E00000005,3,112,E01000001,City of London 001A,E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E09000001,City of London
3,E00000007,3,144,E01000001,City of London 001A,E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E09000001,City of London
4,E00000010,3,178,E01000003,City of London 001C,E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E09000001,City of London


,column,missing_count,missing_pct
4,LSOA21NM,10275,0.0544
3,LSOA21CD,10275,0.0544
7,LAD23CD,10275,0.0544
6,MSOA21NM,10275,0.0544
5,MSOA21CD,10275,0.0544
8,LAD23NM,10275,0.0544
1,cluster_id,0,0.0000
0,OA21CD,0,0.0000
2,population,0,0.0000
9,WD25CD,0,0.0000


# Note: The missing files are for Welsh areas

In [9]:
missing_exact = oa_base[oa_base["LSOA21CD"].isna()].copy()

print("Missing exact lookup rows:", len(missing_exact))
print("Missing OA prefixes:")
print(missing_exact["OA21CD"].str[0].value_counts())

print("\nMissing rows by LAD25NM:")
display(
    missing_exact
    .groupby(["LAD25CD", "LAD25NM"], as_index=False)
    .agg(
        oa_count=("OA21CD", "count"),
        population=("population", "sum")
    )
    .sort_values("population", ascending=False)
    .head(30)
)

Missing exact lookup rows: 10275
Missing OA prefixes:
OA21CD
W    10275
Name: count, dtype: int64

Missing rows by LAD25NM:


,LAD25CD,LAD25NM,oa_count,population
13,W06000015,Cardiff,1115,362289
9,W06000011,Swansea,806,238498
14,W06000016,Rhondda Cynon Taf,789,237670
8,W06000010,Carmarthenshire,615,187876
15,W06000018,Caerphilly,575,175945
19,W06000022,Newport,511,159579
4,W06000005,Flintshire,510,154959
11,W06000013,Bridgend,480,145484
10,W06000012,Neath Port Talbot,485,142312
5,W06000006,Wrexham,446,135121


In [10]:
oa_base["exact_lookup_status"] = np.where(
    oa_base["LSOA21CD"].notna(),
    "exact_lookup_matched",
    "missing_from_england_only_exact_lookup"
)

oa_base["ward_lookup_status"] = np.where(
    oa_base["WD25CD"].notna(),
    "ward_lookup_matched",
    "ward_lookup_missing"
)

oa_base["oa_country_inferred"] = np.select(
    [
        oa_base["OA21CD"].str.startswith("E"),
        oa_base["OA21CD"].str.startswith("W"),
    ],
    [
        "England",
        "Wales",
    ],
    default="Unknown"
)

display(
    oa_base
    .groupby(["oa_country_inferred", "exact_lookup_status"], as_index=False)
    .agg(
        oa_count=("OA21CD", "count"),
        population=("population", "sum")
    )
)

,oa_country_inferred,exact_lookup_status,oa_count,population
0,England,exact_lookup_matched,178605,56490284
1,Wales,missing_from_england_only_exact_lookup,10275,3107463


In [11]:
oa_base.to_csv(OUTPUT_DIR / f"k{K}_oa_geo_cluster_base_v1.csv", index=False)

## 6. Aggregate cluster shares upward

For clusters, we sum OA population by cluster inside each target geography.

Ward aggregation explicitly groups by `LAD25CD`, `LAD25NM`, `WD25CD`, and `WD25NM`, so LAD fields are preserved.

In [13]:
CLUSTER_NAMES = {
    0: "Student & Transient Youth",
    1: "Rooted Older Homeowners",
    2: "Stable Suburban Professionals",
    3: "Cosmopolitan Young Professional Core",
    4: "Settled Working Families / Skilled Trades Suburbs",
    5: "Settled Diverse Urban Communities",
    6: "Post-Industrial Estates / Deprived Working Communities",
}


def add_cluster_names(df):
    df = df.copy()
    df["dominant_cluster_name"] = df["dominant_cluster"].map(CLUSTER_NAMES)
    df["second_cluster_name"] = df["second_cluster"].map(CLUSTER_NAMES)

    for cid, name in CLUSTER_NAMES.items():
        share_col = f"cluster_{cid}_share"
        pop_col = f"cluster_{cid}_population"
        safe_name = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_").lower()

        if share_col in df.columns:
            df[f"{safe_name}_share"] = df[share_col]
        if pop_col in df.columns:
            df[f"{safe_name}_population"] = df[pop_col]

    return df


def aggregate_clusters(
    df,
    target_code,
    target_name=None,
    extra_group_cols=None,
    k=7,
    geography_level=None,
    source_lookup="",
):
    group_cols = build_group_cols(df, target_code, target_name, extra_group_cols)

    working = df.dropna(subset=[target_code, "cluster_id", "population"]).copy()
    working["cluster_id"] = working["cluster_id"].astype(int)

    base = (
        working
        .groupby(group_cols, as_index=False)
        .agg(
            population=("population", "sum"),
            oa_count=("OA21CD", "nunique")
        )
    )

    cluster_pop = (
        working
        .pivot_table(
            index=group_cols,
            columns="cluster_id",
            values="population",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    cluster_pop.columns = [
        f"cluster_{int(c)}_population" if isinstance(c, (int, np.integer)) else c
        for c in cluster_pop.columns
    ]

    out = base.merge(cluster_pop, on=group_cols, how="left")

    for cid in range(k):
        pop_col = f"cluster_{cid}_population"
        if pop_col not in out.columns:
            out[pop_col] = 0

    for cid in range(k):
        out[f"cluster_{cid}_share"] = safe_divide(
            out[f"cluster_{cid}_population"],
            out["population"]
        )

    share_cols = [f"cluster_{cid}_share" for cid in range(k)]

    def top_clusters(row):
        shares = row[share_cols].astype(float)
        ranked = shares.sort_values(ascending=False)

        dominant_cluster = int(ranked.index[0].replace("cluster_", "").replace("_share", ""))
        second_cluster = int(ranked.index[1].replace("cluster_", "").replace("_share", ""))

        return pd.Series({
            "dominant_cluster": dominant_cluster,
            "second_cluster": second_cluster,
            "dominant_cluster_share": ranked.iloc[0],
            "second_cluster_share": ranked.iloc[1],
            "cluster_fragmentation_index": 1 - np.sum(np.square(shares.values))
        })

    out = pd.concat([out, out.apply(top_clusters, axis=1)], axis=1)
    out = add_cluster_names(out)

    return add_area_metadata(
        out,
        geography_level or target_code,
        "population_weighted_oa_cluster_sum",
        source_lookup
    )

In [14]:
cluster_outputs = {}

cluster_outputs["lsoa21"] = aggregate_clusters(
    oa_base,
    target_code="LSOA21CD",
    target_name="LSOA21NM",
    k=K,
    geography_level="LSOA21",
    source_lookup=EXACT_LOOKUP_PATH.name
)

cluster_outputs["msoa21"] = aggregate_clusters(
    oa_base,
    target_code="MSOA21CD",
    target_name="MSOA21NM",
    k=K,
    geography_level="MSOA21",
    source_lookup=EXACT_LOOKUP_PATH.name
)

cluster_outputs["lad23"] = aggregate_clusters(
    oa_base,
    target_code="LAD23CD",
    target_name="LAD23NM",
    k=K,
    geography_level="LAD23",
    source_lookup=EXACT_LOOKUP_PATH.name
)

cluster_outputs["ward25"] = aggregate_clusters(
    oa_base,
    target_code="WD25CD",
    target_name="WD25NM",
    extra_group_cols=["LAD25CD", "LAD25NM"],
    k=K,
    geography_level="WARD25",
    source_lookup=WARD_LOOKUP_PATH.name
)

cluster_outputs["lad25"] = aggregate_clusters(
    oa_base,
    target_code="LAD25CD",
    target_name="LAD25NM",
    k=K,
    geography_level="LAD25",
    source_lookup=WARD_LOOKUP_PATH.name
)

if {"LEPCD", "LEPNM"}.issubset(oa_base.columns):
    cluster_outputs["lep"] = aggregate_clusters(
        oa_base,
        target_code="LEPCD",
        target_name="LEPNM",
        k=K,
        geography_level="LEP",
        source_lookup=EXACT_LOOKUP_PATH.name
    )

for level, df in cluster_outputs.items():
    out_path = OUTPUT_DIR / f"k{K}_{level}_cluster_profile_v1.csv"
    df.to_csv(out_path, index=False)
    print(level, df.shape, "->", out_path.name)

lsoa21 (33755, 42) -> k7_lsoa21_cluster_profile_v1.csv
msoa21 (6856, 42) -> k7_msoa21_cluster_profile_v1.csv
lad23 (296, 42) -> k7_lad23_cluster_profile_v1.csv
ward25 (7572, 44) -> k7_ward25_cluster_profile_v1.csv
lad25 (318, 42) -> k7_lad25_cluster_profile_v1.csv


## 7. Aggregate Census-derived statistics upward

For Census statistics, do not average OA percentages. Sum counts upward, then recalculate percentages from the correct denominators.

In [15]:
oa_stats_base = oa_base.merge(
    features,
    on="OA21CD",
    how="left",
    validate="one_to_one"
)

print("OA stats base shape:", oa_stats_base.shape)

OA stats base shape: (188880, 168)


In [16]:
COUNT_DENOMINATOR_COLS = [
    "total_residents",
    "household_residents",
    "communal_establishment_residents",
    "partnership_total",
    "household_composition_total",
    "country_of_birth_total",
    "length_residence_total",
    "ethnic_group_total",
    "accommodation_total",
    "tenure_total",
    "occupation_total",
    "economic_activity_total",
    "qualification_total",
]


def aggregate_counts(
    df,
    target_code,
    target_name=None,
    extra_group_cols=None,
    geography_level=None,
    source_lookup="",
):
    group_cols = build_group_cols(df, target_code, target_name, extra_group_cols)

    geography_cols = {
        "OA21CD", "cluster_id", "population",
        "LSOA21CD", "LSOA21NM", "MSOA21CD", "MSOA21NM",
        "LAD23CD", "LAD23NM", "LEPCD", "LEPNM",
        "WD25CD", "WD25NM", "LAD25CD", "LAD25NM",
        "geography_level", "aggregation_method", "source_lookup"
    }

    count_cols = [
        c for c in df.columns
        if (
            c.endswith("_count")
            or c in COUNT_DENOMINATOR_COLS
        )
        and c not in geography_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    working = df.dropna(subset=[target_code]).copy()

    out = working.groupby(group_cols, as_index=False)[count_cols].sum()

    return add_area_metadata(
        out,
        geography_level or target_code,
        "oa_count_sum_then_recalculate_percentages",
        source_lookup
    )


def add_recalculated_percentages(df):
    df = df.copy()

    def add_pct(new_col, num_col, den_col):
        if {num_col, den_col}.issubset(df.columns):
            df[new_col] = safe_divide(df[num_col], df[den_col])

    # Age
    add_pct("age_0_14_pct", "age_0_14_count", "total_residents")
    add_pct("age_15_24_pct", "age_15_24_count", "total_residents")
    add_pct("age_25_34_pct", "age_25_34_count", "total_residents")
    add_pct("age_35_49_pct", "age_35_49_count", "total_residents")
    add_pct("age_50_64_pct", "age_50_64_count", "total_residents")
    add_pct("age_65_plus_pct", "age_65_plus_count", "total_residents")

    # Rootedness / migration
    add_pct("uk_born_pct", "uk_born_count", "country_of_birth_total")
    add_pct("non_uk_born_pct", "non_uk_born_count", "country_of_birth_total")
    add_pct("resident_10_plus_years_pct", "resident_10_plus_years_count", "length_residence_total")
    add_pct("resident_less_5_years_pct", "resident_less_5_years_count", "length_residence_total")

    # Ethnicity
    add_pct("white_british_pct", "white_british_count", "ethnic_group_total")
    add_pct("white_other_pct", "white_other_count", "ethnic_group_total")
    add_pct("non_white_pct", "non_white_count", "ethnic_group_total")

    # Housing / tenure
    add_pct("owned_pct", "owned_count", "tenure_total")
    add_pct("owns_outright_pct", "owns_outright_count", "tenure_total")
    add_pct("owns_mortgage_pct", "owns_mortgage_count", "tenure_total")
    add_pct("social_rented_pct", "social_rented_count", "tenure_total")
    add_pct("private_rented_pct", "private_rented_count", "tenure_total")
    add_pct("house_type_pct", "house_type_count", "accommodation_total")
    add_pct("flat_type_pct", "flat_type_count", "accommodation_total")

    # Occupation
    add_pct("managerial_professional_pct", "managerial_professional_count", "occupation_total")
    add_pct("skilled_traditional_pct", "skilled_traditional_count", "occupation_total")
    add_pct("routine_service_elementary_pct", "routine_service_elementary_count", "occupation_total")

    # Economic activity
    add_pct("employed_pct", "employed_count", "economic_activity_total")
    add_pct("unemployed_pct", "unemployed_count", "economic_activity_total")
    add_pct("full_time_student_pct", "full_time_student_count", "economic_activity_total")
    add_pct("retired_pct", "retired_count", "economic_activity_total")
    add_pct("long_term_sick_disabled_pct", "long_term_sick_disabled_count", "economic_activity_total")

    # Education
    add_pct("no_qualifications_pct", "no_qualifications_count", "qualification_total")
    add_pct("level_1_2_pct", "level_1_2_count", "qualification_total")
    add_pct("apprenticeship_pct", "apprenticeship_count", "qualification_total")
    add_pct("level_4_plus_pct", "level_4_plus_count", "qualification_total")

    # Household composition
    add_pct("one_person_household_pct", "one_person_household_count", "household_composition_total")
    add_pct("married_couple_family_pct", "married_couple_family_count", "household_composition_total")
    add_pct("lone_parent_family_pct", "lone_parent_family_count", "household_composition_total")

    return df

In [17]:
stats_outputs = {}

stats_outputs["lsoa21"] = add_recalculated_percentages(
    aggregate_counts(
        oa_stats_base,
        target_code="LSOA21CD",
        target_name="LSOA21NM",
        geography_level="LSOA21",
        source_lookup=EXACT_LOOKUP_PATH.name
    )
)

stats_outputs["msoa21"] = add_recalculated_percentages(
    aggregate_counts(
        oa_stats_base,
        target_code="MSOA21CD",
        target_name="MSOA21NM",
        geography_level="MSOA21",
        source_lookup=EXACT_LOOKUP_PATH.name
    )
)

stats_outputs["lad23"] = add_recalculated_percentages(
    aggregate_counts(
        oa_stats_base,
        target_code="LAD23CD",
        target_name="LAD23NM",
        geography_level="LAD23",
        source_lookup=EXACT_LOOKUP_PATH.name
    )
)

stats_outputs["ward25"] = add_recalculated_percentages(
    aggregate_counts(
        oa_stats_base,
        target_code="WD25CD",
        target_name="WD25NM",
        extra_group_cols=["LAD25CD", "LAD25NM"],
        geography_level="WARD25",
        source_lookup=WARD_LOOKUP_PATH.name
    )
)

stats_outputs["lad25"] = add_recalculated_percentages(
    aggregate_counts(
        oa_stats_base,
        target_code="LAD25CD",
        target_name="LAD25NM",
        geography_level="LAD25",
        source_lookup=WARD_LOOKUP_PATH.name
    )
)

if {"LEPCD", "LEPNM"}.issubset(oa_stats_base.columns):
    stats_outputs["lep"] = add_recalculated_percentages(
        aggregate_counts(
            oa_stats_base,
            target_code="LEPCD",
            target_name="LEPNM",
            geography_level="LEP",
            source_lookup=EXACT_LOOKUP_PATH.name
        )
    )

for level, df in stats_outputs.items():
    out_path = OUTPUT_DIR / f"k{K}_{level}_stats_profile_v1.csv"
    df.to_csv(out_path, index=False)
    print(level, df.shape, "->", out_path.name)

lsoa21 (33755, 122) -> k7_lsoa21_stats_profile_v1.csv
msoa21 (6856, 122) -> k7_msoa21_stats_profile_v1.csv
lad23 (296, 122) -> k7_lad23_stats_profile_v1.csv
ward25 (7572, 124) -> k7_ward25_stats_profile_v1.csv
lad25 (318, 122) -> k7_lad25_stats_profile_v1.csv


## 8. Combine cluster profiles and statistics profiles

The ward merge uses LAD + ward keys to prevent losing or misaligning `LAD25CD` and `LAD25NM`.

In [18]:
JOIN_KEYS = {
    "lsoa21": ["LSOA21CD", "LSOA21NM"],
    "msoa21": ["MSOA21CD", "MSOA21NM"],
    "lad23": ["LAD23CD", "LAD23NM"],
    "ward25": ["LAD25CD", "LAD25NM", "WD25CD", "WD25NM"],
    "lad25": ["LAD25CD", "LAD25NM"],
    "lep": ["LEPCD", "LEPNM"],
}

full_outputs = {}

for level in cluster_outputs.keys():
    if level not in stats_outputs:
        continue

    keys = JOIN_KEYS[level]
    cluster_df = cluster_outputs[level]
    stats_df = stats_outputs[level]

    stats_df_merge = stats_df.drop(
        columns=["geography_level", "aggregation_method", "source_lookup"],
        errors="ignore"
    )

    final = cluster_df.merge(
        stats_df_merge,
        on=keys,
        how="left",
        validate="one_to_one"
    )

    full_outputs[level] = final

    out_path = OUTPUT_DIR / f"k{K}_{level}_full_profile_v1.csv"
    final.to_csv(out_path, index=False)
    print(level, final.shape, "->", out_path.name)

lsoa21 (33755, 159) -> k7_lsoa21_full_profile_v1.csv
msoa21 (6856, 159) -> k7_msoa21_full_profile_v1.csv
lad23 (296, 159) -> k7_lad23_full_profile_v1.csv
ward25 (7572, 161) -> k7_ward25_full_profile_v1.csv
lad25 (318, 159) -> k7_lad25_full_profile_v1.csv


## 9. Derived ward files: enriched, map-ready and North West subset

These are the main working files for ward-level mapping and later political analysis.

In [20]:
if "ward25" in full_outputs:
    ward_final = full_outputs["ward25"].copy()

    required_ward_cols = ["LAD25CD", "LAD25NM", "WD25CD", "WD25NM"]
    missing = [c for c in required_ward_cols if c not in ward_final.columns]
    if missing:
        raise ValueError(f"Ward final output is missing required geography columns: {missing}")

    ward_final["is_mixed_ward"] = ward_final["dominant_cluster_share"] < 0.40
    ward_final["is_clear_dominant_ward"] = ward_final["dominant_cluster_share"] >= 0.60
    ward_final["is_highly_fragmented"] = ward_final["cluster_fragmentation_index"] >= 0.70

    ward_enriched_path = OUTPUT_DIR / f"k{K}_ward25_named_full_profile_v1_enriched.csv"
    ward_final.to_csv(ward_enriched_path, index=False)
    print("Saved:", ward_enriched_path.name)

    ward_map_cols = [
        "LAD25CD", "LAD25NM", "WD25CD", "WD25NM",
        "population", "oa_count",
        "dominant_cluster", "dominant_cluster_name", "dominant_cluster_share",
        "second_cluster", "second_cluster_name", "second_cluster_share",
        "cluster_fragmentation_index",
        "is_mixed_ward", "is_clear_dominant_ward", "is_highly_fragmented",
    ]

    existing_map_cols = [c for c in ward_map_cols if c in ward_final.columns]
    ward_map = ward_final[existing_map_cols].copy()

    ward_map_path = OUTPUT_DIR / f"k{K}_ward25_map_ready_v1.csv"
    ward_map.to_csv(ward_map_path, index=False)
    print("Saved:", ward_map_path.name)

    north_west_lads = [
        "Cheshire East", "Cheshire West and Chester", "Halton", "Warrington",
        "Cumberland", "Westmorland and Furness",
        "Bolton", "Bury", "Manchester", "Oldham", "Rochdale", "Salford",
        "Stockport", "Tameside", "Trafford", "Wigan",
        "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
        "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
        "Rossendale", "South Ribble", "West Lancashire", "Wyre",
        "Knowsley", "Liverpool", "Sefton", "St. Helens", "Wirral",
    ]

    ward_nw = ward_final[ward_final["LAD25NM"].isin(north_west_lads)].copy()
    ward_nw["north_west_subset"] = True
    ward_nw["boundary_note"] = np.where(
        ward_nw["LAD25NM"].eq("Sefton"),
        "Sefton changed for 2026; WD25 is not suitable for final 2026/2027 ward mapping.",
        "WD25 used as current working geography."
    )

    ward_nw_path = OUTPUT_DIR / f"k{K}_north_west_ward25_full_profile_v1.csv"
    ward_nw.to_csv(ward_nw_path, index=False)
    print("Saved:", ward_nw_path.name)

Saved: k7_ward25_named_full_profile_v1_enriched.csv
Saved: k7_ward25_map_ready_v1.csv
Saved: k7_north_west_ward25_full_profile_v1.csv


## 10. Sanity checks

These checks catch the most common silent problems: missing LAD fields, cluster shares not summing to 1, or unexpected population loss.

In [21]:
print("Total OA population in assignments:", assignments["population"].sum())

for level, df in cluster_outputs.items():
    print(level, "aggregated population:", df["population"].sum(), "rows:", len(df))

print("\\nCluster-share sum checks:")
for level, df in cluster_outputs.items():
    share_cols = [f"cluster_{i}_share" for i in range(K) if f"cluster_{i}_share" in df.columns]
    check = df[share_cols].sum(axis=1)
    print(level)
    print(check.describe())

if "ward25" in full_outputs:
    print("\\nWard LAD coverage:")
    print(full_outputs["ward25"][["LAD25CD", "LAD25NM"]].isna().sum())

outputs = sorted(OUTPUT_DIR.glob("*.csv"))
print("\\nGenerated CSVs:")
for p in outputs:
    print(p.name)

Total OA population in assignments: 59597747
lsoa21 aggregated population: 56490284 rows: 33755
msoa21 aggregated population: 56490284 rows: 6856
lad23 aggregated population: 56490284 rows: 296
ward25 aggregated population: 59597747 rows: 7572
lad25 aggregated population: 59597747 rows: 318
\nCluster-share sum checks:
lsoa21
count    3.375500e+04
mean     1.000000e+00
std      1.583893e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
msoa21
count    6.856000e+03
mean     1.000000e+00
std      3.737808e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
lad23
count    2.960000e+02
mean     1.000000e+00
std      5.959484e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
ward25
count    7.572000e+03
mean     1.000000e+00
std      3.649306e-17
min      1.000